# Train Identifier

Self-contained workflow for training the retrieval model and building the FAISS index from `confirmed_pairs.jsonl`.


## Bootstrap Repository


In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys, urllib.request

REPO_URL = 'https://github.com/eftpmc/reddibase.git'
REPO_DIR = Path.home() / 'reddibase'

def in_repo(path):
    return (path / 'requirements.txt').exists() and (path / 'framework').exists()

cwd = Path.cwd()
if in_repo(cwd):
    ROOT = cwd
else:
    if not REPO_DIR.exists():
        subprocess.check_call(['git', 'clone', REPO_URL, str(REPO_DIR)])
    ROOT = REPO_DIR

os.chdir(ROOT)
print(f'Repo root: {Path.cwd()}')
print(f'Python: {sys.executable}')


## Install Dependencies


In [ ]:
%pip install -r requirements.txt


## GPU Check


In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('CUDA version:', torch.version.cuda)
else:
    print('No CUDA GPU found. This notebook will work, but full training will be slow.')


## Configuration


In [ ]:
MODEL = 'tipofmyjoystick'
PAIRS = Path(f'models/{MODEL}/confirmed_pairs.jsonl')
OUTPUT = Path(f'models/{MODEL}/identification_model')
HF_MODEL_REPO = 'eftpmc/tipofmyjoystick-identification'
EPOCHS = 3
BATCH_SIZE = 128
print('Pairs:', PAIRS)
print('Output:', OUTPUT)
print('Model Hub repo:', HF_MODEL_REPO)


## Acquire Pairs Artifact

This notebook requires `confirmed_pairs.jsonl`. If it is not already in the repo, set `PAIRS_SOURCE` to a local path or URL and run this cell.


In [ ]:
PAIRS_SOURCE = ''
PAIRS.parent.mkdir(parents=True, exist_ok=True)
if not PAIRS.exists() and PAIRS_SOURCE:
    if PAIRS_SOURCE.startswith(('http://', 'https://')):
        print(f'Downloading {PAIRS_SOURCE} -> {PAIRS}')
        urllib.request.urlretrieve(PAIRS_SOURCE, PAIRS)
    else:
        src = Path(PAIRS_SOURCE).expanduser()
        if not src.exists():
            raise FileNotFoundError(f'PAIRS_SOURCE does not exist: {src}')
        print(f'Copying {src} -> {PAIRS}')
        shutil.copyfile(src, PAIRS)
if not PAIRS.exists():
    raise FileNotFoundError(f'Missing {PAIRS}. Run the classifier notebook first, or set PAIRS_SOURCE above.')
pair_count = sum(1 for _ in PAIRS.open(encoding='utf-8'))
print(f'Confirmed pairs: {pair_count:,}')


## Train Identifier + Build Index


In [ ]:
!python -m scripts.train_embedder {MODEL} --pairs {PAIRS} --output {OUTPUT} --epochs {EPOCHS} --batch-size {BATCH_SIZE}


## Artifact Check


In [ ]:
for name in ['index.faiss', 'pairs.jsonl']:
    path = OUTPUT / name
    print(name, path.exists(), f'{path.stat().st_size / (1024**2):.2f} MB' if path.exists() else '')


## Identifier Sanity Check

This quick check uses the first confirmed pair as a query and verifies the trained index can retrieve that same source near the top. It is not a full benchmark, but it catches broken indexes, mismatched pair files, and obviously bad encoder saves.


In [ ]:
import json
from framework.embedder import IdentificationModel

with PAIRS.open(encoding='utf-8') as f:
    probe = json.loads(next(f))

model = IdentificationModel(str(OUTPUT), str(OUTPUT)).load()
results = model.search(probe['description'], top_k=10)
source_ids = [pair.source_id for pair, _score in results]
print('Probe source:', probe.get('source_id') or probe.get('post_id'))
print('Top sources:', source_ids)
if (probe.get('source_id') or probe.get('post_id')) not in source_ids:
    raise AssertionError('Probe pair was not retrieved in the top 10; inspect the trained identifier before pushing.')


## Create Hugging Face Identifier Repo

Run this once per target repo. Set `private=True` if you want to review the artifacts before publishing.


In [ ]:
from huggingface_hub import create_repo

create_repo(HF_MODEL_REPO, repo_type='model', exist_ok=True, private=False)
print(f'Ready: https://huggingface.co/{HF_MODEL_REPO}')


## Push Identifier Artifacts To Hugging Face


In [ ]:
from huggingface_hub import upload_folder

required = ['index.faiss', 'pairs.jsonl', 'modules.json']
missing = [name for name in required if not (OUTPUT / name).exists()]
if missing:
    raise FileNotFoundError(f'Missing identifier artifacts: {missing}')

upload_folder(
    repo_id=HF_MODEL_REPO,
    repo_type='model',
    folder_path=str(OUTPUT),
    commit_message=f'Upload {MODEL} identifier artifacts',
)
print(f'Pushed identifier artifacts: https://huggingface.co/{HF_MODEL_REPO}')
